In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
# Seed for reproducibility
np.random.seed(42)

In [3]:
# Generate linearly correlated feature data
n_samples = 100
x1 = np.random.rand(n_samples)  # Feature 1
x2 = 2 * x1    # Feature 2, highly correlated with Feature 1 with no additional noise

In [4]:
# Create a target y with some noise
y = 3 * x1 + 5 * x2 + np.random.randn(n_samples) * 0.5  # y = 3*X1 + 5*X2 + noise

In [5]:
# Define a range of values for beta_1 and beta_2 to calculate the loss surface
beta_1_range = np.linspace(-10, 10, 200)
beta_2_range = np.linspace(-20, 20, 200)
beta_1, beta_2 = np.meshgrid(beta_1_range, beta_2_range)

In [6]:
# Calculate OLS and Ridge loss surfaces for each combination of beta_1 and beta_2
def calculate_loss(beta_1, beta_2, x1, x2, y, lambda_ridge=0):
    loss = np.zeros(beta_1.shape)
    for i in range(beta_1.shape[0]):
        for j in range(beta_1.shape[1]):
            y_pred = beta_1[i, j] * x1 + beta_2[i, j] * x2
            loss[i, j] = (1/len(y)) * np.sum((y - y_pred) ** 2) + lambda_ridge * (beta_1[i, j]**2 + beta_2[i, j]**2)
    return loss

In [7]:
loss_OLS = calculate_loss(beta_1, beta_2, x1, x2, y)
loss_Ridge = calculate_loss(beta_1, beta_2, x1, x2, y, lambda_ridge=500)

In [8]:
# Define the stagnation line where beta_2 = 2 * beta_1
stagnation_beta_1 = beta_1_range
stagnation_beta_2 = 2 * stagnation_beta_1

In [ ]:
# OLS surface plot
fig_OLS = go.Figure(data=[go.Surface(z=loss_OLS, x=beta_1, y=beta_2, colorscale='Viridis', opacity=0.8)])
fig_OLS.update_layout(title='OLS Loss Surface', scene=dict(
    xaxis_title='Beta 1',
    yaxis_title='Beta 2',
    zaxis_title='Loss'
))
# fig_OLS.add_trace(go.Scatter3d(x=stagnation_beta_1, y=stagnation_beta_2, z=[np.min(loss_OLS)] * len(stagnation_beta_1), mode='lines', line=dict(color='red', width=5), name='Stagnation Line'))
fig_OLS.show()

# Ridge surface plot
fig_Ridge = go.Figure(data=[go.Surface(z=loss_Ridge, x=beta_1, y=beta_2, colorscale='Viridis', opacity=0.8)])
fig_Ridge.update_layout(title='Ridge Loss Surface', scene=dict(
    xaxis_title='Beta 1',
    yaxis_title='Beta 2',
    zaxis_title='Loss'
))
# fig_Ridge.add_trace(go.Scatter3d(x=stagnation_beta_1, y=stagnation_beta_2, z=[np.min(loss_Ridge)] * len(stagnation_beta_1), mode='lines', line=dict(color='red', width=5), name='Stagnation Line'))
fig_Ridge.show()

: 

On observing OLS loss surface, at the bottom, there isnot minima as it not a point, but there is aset of minimum values (a line of values). There there are huge number of points where loss function is minimum. This is huge problem as there will be huge number of beta_1 and beta_2 which might lead to correct answer, which is not acceptable, as there should be only one set of acceptable answer.

##### This loss curve surface with flat bottom is called RIDGE
At flat bottom, the loss function is same, here horizontal. Here, we have infinite values of beta_1 and beta_2. This is one of many problem. Even if we change the beta values, the loss will not change since this is flat at bottom and extends till infinity. And, so, sometimetimes we get solutions with very high parameter values because even those will lead to same amount of loss.

When we add the regularization term, it lifts the flat surface, and makes it curved, giving the actual single value of local minima.

#### Hence, when the features are corelated (W.T * W does not exits), the loss function function looks like the OLS loss, making ridge at the bottom, giving inifinite values of beta_1 and beta_2 which cannot lead to correct answer. This means our potential optimum can have very high values, which is not good.